In [1]:
import sqlite3

def main():
    conn = sqlite3.connect("library.db")
    curr = conn.cursor()

    curr.execute('''CREATE TABLE IF NOT EXISTS books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        quantity INTEGER NOT NULL
    )''')

    books_data = [
        ("Life Stories", 1),
        ("Butter Recipes", 2),
        ("Food Cooks", 3),
        ("Dessert Plans", 4),
        ("Breakfast Foods", 5)
    ]

    curr.execute("SELECT COUNT(*) FROM books")

    if curr.fetchone()[0] == 0:
        curr.executemany('''INSERT INTO books (title, quantity)
        VALUES (?, ?)
        ''', books_data)

    curr.execute('''CREATE TABLE IF NOT EXISTS borrowed_books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_name TEXT NOT NULL,
        book_id INTEGER NOT NULL,
        borrow_date TEXT,
        return_date TEXT,
        fine REAL DEFAULT 0,
        FOREIGN KEY (book_id) REFERENCES books(id)
    )''')

    curr.execute("SELECT COUNT(*) FROM borrowed_books")

    if curr.fetchone()[0] == 0:
        curr.execute('''INSERT INTO borrowed_books
        (student_name, book_id, borrow_date, return_date, fine)
        VALUES (?, ?, ?, ?, ?)
        ''', ("Sample Student", 1, "12/12/2025", "22/12/2025", 1.0))

    conn.commit()
    conn.close()

main()

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from tkcalendar import DateEntry
from datetime import date, datetime
import sqlite3


LIBRARY_DB_PATH = "library.db"


def connect_db():

    return sqlite3.connect(LIBRARY_DB_PATH)

def update_total_count(count):

    try:
        total_count_label.config(text=f"Total Records: {count}")
    except tk.TclError:
        pass  

def updateBookList():

    conn = connect_db()
    curr = conn.cursor()
    curr.execute("SELECT DISTINCT title FROM books WHERE quantity > 0 ORDER BY title COLLATE NOCASE")
    titles = [row[0] for row in curr.fetchall()]
    conn.close()


    seen = []
    for title in titles:
        if title not in seen:
            seen.append(title)
    book_combobox["values"] = seen

def refresh_table():

    for row in record_view.get_children():
        record_view.delete(row)

    conn = connect_db()
    curr = conn.cursor()
    curr.execute('''
        SELECT bb.id, bb.student_name, b.title, bb.borrow_date, bb.return_date, bb.fine
        FROM borrowed_books bb
        JOIN books b ON bb.book_id = b.id
        ORDER BY bb.id DESC
    ''')
    rows = curr.fetchall()
    conn.close()


    for row in rows:
        fine = row[5]

        tag = 'late' if fine > 0 else ''
        record_view.insert("", "end", values=row, tags=(tag,))


    update_total_count(len(rows))


def validate_dates(borrow_str, return_str):

    try:
        today = date.today()
        borrow_date = datetime.strptime(borrow_str, "%m/%d/%y").date()
        return_date = datetime.strptime(return_str, "%m/%d/%y").date()


        if borrow_date < today:
            messagebox.showwarning("Date Error", "Borrow Date cannot be in the past.")
            return False, None, None


        if return_date < borrow_date:
            messagebox.showwarning("Date Error", "Return Date cannot be before the Borrow Date.")
            return False, None, None


        return True, borrow_date, return_date

    except ValueError:
        messagebox.showerror("Date Format Error", "Invalid date format. Use MM/DD/YY.")
        return False, None, None


def add_record():

    try:
        student = studentEntry.get().strip()
        book_title = book_combobox.get().strip()
        borrow_str = borrow_date_entry.get()
        return_str = return_date_entry.get()
        today = date.today()


        if not (student and book_title and borrow_str and return_str):
            messagebox.showwarning("Missing Info", "Please fill in all fields.")
            return


        valid, b_obj, r_obj = validate_dates(borrow_str, return_str)
        if not valid:
            return

        conn = connect_db()
        curr = conn.cursor()


        curr.execute("SELECT id, quantity FROM books WHERE LOWER(title)=LOWER(?)", (book_title,))
        result = curr.fetchone()


        if not result:
            messagebox.showerror("Error", "Book not found.")
            conn.close()
            return

        book_id, quantity = result

        if quantity <= 0:
            messagebox.showwarning("Unavailable", f"'{book_title}' is out of stock.")
            conn.close()
            return


        fine = 0.0
        if today > r_obj:
            fine = (today - r_obj).days * 5.0

 
        curr.execute('''
            INSERT INTO borrowed_books (student_name, book_id, borrow_date, return_date, fine)
            VALUES (?, ?, ?, ?, ?)
        ''', (student, book_id, b_obj.strftime('%Y-%m-%d'), r_obj.strftime('%Y-%m-%d'), fine))


        curr.execute("UPDATE books SET quantity = quantity - 1 WHERE id = ?", (book_id,))
        conn.commit()
        conn.close()

        messagebox.showinfo("Success", "Record added successfully.")
        refresh_table()
        updateBookList()
        clear_fields(reset_dates=False)

    except Exception as e:
        messagebox.showerror("Error", f"Failed to add record: {e}")

def delete_record():

    selected = record_view.selection()
    if not selected:
        messagebox.showwarning("No selection", "Please select a record to delete.")
        return

    confirm = messagebox.askyesno("Confirm Delete", "Delete this record and increase quantity?")
    if not confirm:
        return

    try:
        conn = connect_db()
        curr = conn.cursor()

        record = record_view.item(selected)["values"]
        borrow_id = record[0]
        book_title = record[2]


        curr.execute("SELECT id FROM books WHERE LOWER(title)=LOWER(?)", (book_title,))
        book_id = curr.fetchone()[0]


        curr.execute("DELETE FROM borrowed_books WHERE id = ?", (borrow_id,))
        curr.execute("UPDATE books SET quantity = quantity + 1 WHERE id = ?", (book_id,))
        conn.commit()
        conn.close()

        messagebox.showinfo("Deleted", "Record deleted successfully.")
        refresh_table()
        updateBookList()

    except Exception as e:
        messagebox.showerror("Error", f"Failed to delete record: {e}")

def clear_fields(reset_dates=True):
    
    studentEntry.delete(0, tk.END)
    book_combobox.set("")
    if reset_dates:
        borrow_date_entry.set_date(date.today())
        return_date_entry.set_date(date.today())

def search_records(event=None):
    
    query = search_entry.get().strip().lower()

    for row in record_view.get_children():
        record_view.delete(row)

    conn = connect_db()
    curr = conn.cursor()

    sql = '''
        SELECT bb.id, bb.student_name, b.title, bb.borrow_date, bb.return_date, bb.fine
        FROM borrowed_books bb
        JOIN books b ON bb.book_id = b.id
    '''
    params = []

    if query:
        sql += " WHERE LOWER(bb.student_name) LIKE ? OR LOWER(b.title) LIKE ?"
        params = [f"%{query}%", f"%{query}%"]

    sql += " ORDER BY bb.id DESC"

    curr.execute(sql, params)
    rows = curr.fetchall()
    conn.close()

    for row in rows:
        fine = row[5]
        tag = 'late' if fine > 0 else ''
        record_view.insert("", "end", values=row, tags=(tag,))

    update_total_count(len(rows))

window = tk.Tk()
window.title("Student Book Tracker")
window.geometry("1000x600")
window.config(bg="#bfbfbf")  

left_frame = tk.LabelFrame(window, text="Borrow Book", bg="light grey", padx=10, pady=10)
left_frame.pack(side="left", fill="y", padx=10, pady=10)

tk.Label(left_frame, text="Student Name:", bg="#e0e0e0").grid(row=0, column=0, sticky="w", pady=10)
studentEntry = tk.Entry(left_frame, width=25)
studentEntry.grid(row=0, column=1, pady=10)

tk.Label(left_frame, text="Book:", bg="#e0e0e0").grid(row=1, column=0, sticky="w", pady=10)
book_combobox = ttk.Combobox(left_frame, width=23, state="readonly")
book_combobox.grid(row=1, column=1, pady=10)
updateBookList()

tk.Label(left_frame, text="Borrow Date:", bg="#e0e0e0").grid(row=2, column=0, sticky="w", pady=10)
borrow_date_entry = DateEntry(left_frame, date_pattern='mm/dd/yy', width=22)
borrow_date_entry.grid(row=2, column=1, pady=10)

tk.Label(left_frame, text="Return Date:", bg="#e0e0e0").grid(row=3, column=0, sticky="w", pady=10)
return_date_entry = DateEntry(left_frame, date_pattern='mm/dd/yy', width=22)
return_date_entry.grid(row=3, column=1, pady=10)

add_btn = tk.Button(left_frame, text="Add", bg="green", fg="white", width=8, command=add_record)
add_btn.grid(row=4, column=0, pady=20)

delete_btn = tk.Button(left_frame, text="Delete", bg="red", fg="white", width=8, command=delete_record)
delete_btn.grid(row=4, column=1, pady=20)

clear_btn = tk.Button(left_frame, text="Clear", bg="gray", fg="white", width=8, command=lambda: clear_fields(True))
clear_btn.grid(row=5, column=0, columnspan=2, pady=5)

right_frame = tk.Frame(window, bg="light grey")
right_frame.pack(side="right", fill="both", expand=True, padx=10, pady=10)

top_bar_frame = tk.Frame(right_frame, bg="light grey")
top_bar_frame.pack(fill="x", pady=5)

tk.Label(top_bar_frame, text="Search:", bg="light grey", font=("Arial", 10, "bold")).pack(side="left", padx=5)
search_entry = tk.Entry(top_bar_frame, width=30)
search_entry.pack(side="left", padx=5)
search_entry.bind("<KeyRelease>", search_records)

total_count_label = tk.Label(top_bar_frame, text="Total Records: 0", bg="light grey", font=("Arial", 10, "bold"))
total_count_label.pack(side="right", padx=5)

table_container = tk.Frame(right_frame, bg="white")
table_container.pack(fill="both", expand=True)

columns = ("id", "student", "book", "borrow_date", "return_date", "fine")
record_view = ttk.Treeview(table_container, columns=columns, show="headings", selectmode="browse", height=18)

style = ttk.Style()
style.theme_use('clam')  

style.configure("Custom.Treeview.Heading", background="blue", foreground="white", font=('Arial', 10, 'bold'))

# normal white rows
style.configure("Custom.Treeview", font=('Arial', 10), rowheight=24, fieldbackground="white", background="white")

record_view.configure(style="Custom.Treeview")

# setup column names and widths
for col in columns:
    record_view.heading(col, text=col.replace('_', ' ').title())
    if col == "id":
        record_view.column(col, width=50, anchor="center", stretch=False)
    elif col == "fine":
        record_view.column(col, width=80, anchor="center")
    else:
        record_view.column(col, width=160, anchor="w")

# highlights late rows in red
record_view.tag_configure('late', background='#ffcccc')

# scrollbar on the right side
vsb = ttk.Scrollbar(table_container, orient="vertical", command=record_view.yview)
record_view.configure(yscrollcommand=vsb.set)
record_view.pack(side="left", fill="both", expand=True)
vsb.pack(side="right", fill="y")

# load all records at startup
refresh_table()
window.mainloop()
